In [ ]:
# ── Connection — Variable Library VL_SD  ──────────────────────────────────────
_vl = notebookutils.variableLibrary.getLibrary('VL_SALES_ORD')

SILVER_LH_ABFSS    = _vl['SILVER_LH_ABFSS'].strip()
MD_SILVER_LH_ABFSS = _vl['MD_SILVER_LH_ABFSS'].strip()
SILVER_SCHEMA      = 'dbo'   # reads v2 Silver; separate schema for parallel v2 testing

SD_GOLD_LH_ABFSS   = _vl['SD_GOLD_LH_ABFSS'].strip()
GOLD_SCHEMA        = 'dbo'   # writes to separate schema for parallel v2 testing

OPS_LH_ABFSS       = _vl['OPS_LH_ABFSS'].strip()
OPS_SCHEMA         = 'dbo'

PIPELINE_NAME    = 'SO_Gold'
PIPELINE_RUN_ID  = ''   # blank = auto-generate 

StatementMeta(, 3497bedf-6d4c-46fe-8d2a-cc6a6e5f651d, 3, Finished, Available, Finished, False)

In [2]:
%run ./NB_Utils_Silver_v2

StatementMeta(, 3497bedf-6d4c-46fe-8d2a-cc6a6e5f651d, 10, Finished, Available, Finished, True)

NB_Utils_Silver imports loaded
_ops_append + log_metrics ready
reorder_columns() ready
process_table() ready
apply_xrate() ready
apply_unit_conv() ready

── NB_Utils_Silver fully loaded ────────────────────────────────────────────
_apply_enrichments() ready

── NB_Utils_Silver_v2 fully loaded ─────────────────────────────────────────


In [3]:
# ── Pipeline run setup ────────────────────────────────────────────────────────
if not PIPELINE_RUN_ID:
    PIPELINE_RUN_ID = str(uuid.uuid4())

spark.conf.set('spark.sql.parquet.datetimeRebaseModeInWrite', 'CORRECTED')
spark.conf.set('spark.sql.parquet.datetimeRebaseModeInRead',  'CORRECTED')

start_time = datetime.utcnow()
print(f'Pipeline : {PIPELINE_NAME}')
print(f'Run ID   : {PIPELINE_RUN_ID}')
print(f'Started  : {start_time}')

REDUNDANT_COLS = ['Material', 'Plant', 'Customer', 'Ship_To_Party', 'Sales_Org_Name', 'Plant_Name']

def drop_redundant(df):
    to_drop = [c for c in REDUNDANT_COLS if c in df.columns]
    if to_drop:
        print(f'  Dropping redundant cols: {to_drop}')
        return df.drop(*to_drop)
    return df

def write_gold(df, table_name, sk_col):
    gold_path = f'{SD_GOLD_LH_ABFSS}/{GOLD_SCHEMA}/{table_name}'
    df = (df
          .withColumn('_PIPELINE_NAME',   F.lit(PIPELINE_NAME))
          .withColumn('_PIPELINE_RUN_ID', F.lit(PIPELINE_RUN_ID))
          .withColumn('_UPDATED_AT',      F.current_timestamp()))
    rows_in = df.count()
    if not DeltaTable.isDeltaTable(spark, gold_path):
        df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(gold_path)
        spark.sql(f"CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.{table_name} USING DELTA LOCATION '{gold_path}'")
        print(f'  Created {GOLD_SCHEMA}.{table_name} — {rows_in:,} rows')
    else:
        df.createOrReplaceTempView(f'incoming_{table_name}')
        spark.sql(f"""
            MERGE INTO delta.`{gold_path}` AS tgt
            USING incoming_{table_name}    AS src
            ON tgt.{sk_col} = src.{sk_col}
            WHEN MATCHED     THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        rows_in = spark.read.format('delta').load(gold_path).count()
        print(f'  Merged {GOLD_SCHEMA}.{table_name} — {rows_in:,} rows')
    log_metrics(table_name, 'GOLD', {'rows_out': rows_in})
    return rows_in

print('Helpers ready')

StatementMeta(, 3497bedf-6d4c-46fe-8d2a-cc6a6e5f651d, 11, Finished, Available, Finished, False)

Pipeline : SO_Gold
Run ID   : b68be88d-b947-4600-ae3a-18930dc2489a
Started  : 2026-09-21 23:17:16.849765
Helpers ready


In [4]:
# ── Load active Silver rows ───────────────────────────────────────────────────
def silver(table_name):
    return (spark.read.format('delta')
                 .load(f'{SILVER_LH_ABFSS}/{SILVER_SCHEMA}/{table_name}')
                 .filter(F.col('_IS_DELETED') == False)
                 .drop('_IS_DELETED', '_DELETED_AT', '_DUP_GROUP',
                        '_PIPELINE_NAME', '_PIPELINE_RUN_ID'))

vbak = silver('VBAK')
vbap = silver('VBAP')
vbep = silver('VBEP')
likp = silver('LIKP')
lips = silver('LIPS')

print('Silver tables loaded')

StatementMeta(, 3497bedf-6d4c-46fe-8d2a-cc6a6e5f651d, 12, Finished, Available, Finished, False)

Silver tables loaded


In [5]:
print('\nBuilding Fact_Sales_Doc_Item...')

_header_cols = ['Sales_Order_SK', 'Customer_SK', 'Order_Type', 'Sales_Org',
                'Distribution_Channel', 'Division', 'Sales_Office', 'Sales_Group',
                'Payment_Terms', 'Incoterms', 'Sales_Org_Name',
                'SD_Doc_Category', 'Complete_Delivery']

# Prefix all VBAK header cols with 'Header_' to prevent ambiguous-column errors
# when joining with VBAP (e.g. both tables have Division from SPART).
_present  = [c for c in _header_cols if c in vbak.columns]
vbak_slim = vbak.select(['Sales_Order'] + _present)
for _c in _present:
    vbak_slim = vbak_slim.withColumnRenamed(_c, f'Header_{_c}')

fact_sdi = vbap.join(vbak_slim, on='Sales_Order', how='left')

sk_cols   = [c for c in ('Sales_Order_Item_SK', 'Header_Sales_Order_SK',
                          'Header_Customer_SK', 'Material_SK', 'Plant_SK')
             if c in fact_sdi.columns]
pk_cols   = ['Sales_Order', 'Item']
rest_cols = [c for c in fact_sdi.columns
             if c not in sk_cols + pk_cols and not c.startswith('_')]
meta_cols = [c for c in fact_sdi.columns if c.startswith('_')]
fact_sdi  = fact_sdi.select(sk_cols + pk_cols + rest_cols + meta_cols)

fact_sdi = drop_redundant(fact_sdi)
n = write_gold(fact_sdi, 'Fact_Sales_Doc_Item', 'Sales_Order_Item_SK')

StatementMeta(, 3497bedf-6d4c-46fe-8d2a-cc6a6e5f651d, 13, Finished, Available, Finished, False)


Building Fact_Sales_Doc_Item...
  Dropping redundant cols: ['Material', 'Plant', 'Plant_Name']
  Created dbo.Fact_Sales_Doc_Item — 229,449 rows


In [6]:
print('\nBuilding Fact_Schedule_Line...')

sk_cols   = [c for c in ('Sales_Schedule_SK',) if c in vbep.columns]
pk_cols   = ['Sales_Order', 'Item', 'Schedule_Line']
rest_cols = [c for c in vbep.columns if c not in sk_cols + pk_cols and not c.startswith('_')]
meta_cols = [c for c in vbep.columns if c.startswith('_')]
fact_sch  = vbep.select(sk_cols + pk_cols + rest_cols + meta_cols)

n = write_gold(fact_sch, 'Fact_Schedule_Line', 'Sales_Schedule_SK')

StatementMeta(, 3497bedf-6d4c-46fe-8d2a-cc6a6e5f651d, 14, Finished, Available, Finished, False)


Building Fact_Schedule_Line...
  Created dbo.Fact_Schedule_Line — 357,993 rows


In [7]:
print('\nBuilding Fact_Delivery_Item...')

_likp_header_cols = ['Delivery_SK', 'Customer_SK', 'Document_Date', 'Delivery_Date',
                     'Ship_To_Party', 'Sales_Org', 'Shipping_Point', 'Delivery_Type',
                     'Total_Weight', 'Created_On', 'Incoterms',
                     'Planned_GI_Date', 'Actual_GI_Date']

# Prefix all LIKP header cols with 'Header_' to prevent ambiguous-column errors
# (Total_Weight can appear at both header and item level in SAP).
_likp_present = [c for c in _likp_header_cols if c in likp.columns]
likp_slim = likp.select(['Delivery'] + _likp_present)
for _c in _likp_present:
    likp_slim = likp_slim.withColumnRenamed(_c, f'Header_{_c}')

fact_di = lips.join(likp_slim, on='Delivery', how='left')

# Derive OTIF on-time flag — guard against missing GI date columns in the extract.
_agi = 'Header_Actual_GI_Date'
_pgi = 'Header_Planned_GI_Date'
_has_gi = _agi in fact_di.columns and _pgi in fact_di.columns
if _has_gi:
    fact_di = fact_di.withColumn('On_Time_Delivery',
        F.when(F.col(_agi).isNull(), 'Pending')
         .when(F.col(_agi) <= F.col(_pgi), 'On Time')
         .otherwise('Late'))
    calc_cols = ['On_Time_Delivery'] + [c for c in (_pgi, _agi) if c in fact_di.columns]
else:
    _missing = [c for c in (_agi, _pgi) if c not in fact_di.columns]
    print(f'  On_Time_Delivery skipped — columns not in extract: {_missing}')
    calc_cols = [c for c in (_pgi, _agi) if c in fact_di.columns]

sk_cols   = [c for c in ('Delivery_Item_SK', 'Header_Delivery_SK',
                          'Header_Customer_SK', 'Material_SK', 'Plant_SK')
             if c in fact_di.columns]
# LIPS renames SAP POSNR to Delivery_Item (not Item)
pk_cols   = ['Delivery', 'Delivery_Item']
rest_cols = [c for c in fact_di.columns
             if c not in sk_cols + pk_cols + calc_cols and not c.startswith('_')]
meta_cols = [c for c in fact_di.columns if c.startswith('_')]
fact_di   = fact_di.select(sk_cols + pk_cols + calc_cols + rest_cols + meta_cols)

fact_di = drop_redundant(fact_di)
n = write_gold(fact_di, 'Fact_Delivery_Item', 'Delivery_Item_SK')

StatementMeta(, 3497bedf-6d4c-46fe-8d2a-cc6a6e5f651d, 15, Finished, Available, Finished, False)


Building Fact_Delivery_Item...
  On_Time_Delivery skipped — columns not in extract: ['Header_Planned_GI_Date']
  Dropping redundant cols: ['Material', 'Plant', 'Plant_Name']
  Created dbo.Fact_Delivery_Item — 70,894 rows


In [8]:
# ── Summary ───────────────────────────────────────────────────────────────────
duration = (datetime.utcnow() - start_time).total_seconds()
print('\n' + '=' * 55)
print('SO GOLD — SUMMARY')
print('=' * 55)
for tbl in ('Fact_Sales_Doc_Item', 'Fact_Schedule_Line', 'Fact_Delivery_Item'):
    path = f'{SD_GOLD_LH_ABFSS}/{GOLD_SCHEMA}/{tbl}'
    try:
        n = spark.read.format('delta').load(path).count()
        print(f'  {tbl:<25} {n:>10,}')
    except Exception:
        print(f'  {tbl:<25}  NOT FOUND')
print('-' * 55)
print(f'Run ID   : {PIPELINE_RUN_ID}')
print(f'Duration : {duration:.1f}s')

StatementMeta(, 3497bedf-6d4c-46fe-8d2a-cc6a6e5f651d, 16, Finished, Available, Finished, False)


SO GOLD — SUMMARY
  Fact_Sales_Doc_Item          229,449
  Fact_Schedule_Line           357,993
  Fact_Delivery_Item            70,894
-------------------------------------------------------
Run ID   : b68be88d-b947-4600-ae3a-18930dc2489a
Duration : 63.7s
